# 03 — Exploratory Data Analysis & Visualizations

This notebook explores the scraped review data and produces the figures that appear in the final report. Three questions drive the analysis:

1. How do sentiments in movie reviews vary across genres?
2. How do reviews differ between professional critics and general users?
3. Is there a correlation between critic ratings and audience ratings?

## What this notebook covers
- Rating distributions across genres (users vs. professionals)
- Review volume comparison
- Descriptive statistics
- Sentiment distribution using TextBlob (as an EDA baseline; the full classification work lives in `02_sentiment_analysis.ipynb`)
- Correlation of critic and audience ratings per genre

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob

USER_DIR = "../data/user_reviews"
PRO_DIR  = "../data/professional_reviews"
GENRES = ["adventure", "comedy", "crime", "horror", "superhero"]

sns.set_style("whitegrid")


In [ ]:
def load_data(file_path):
    return pd.read_csv(file_path)

# User reviews (IMDb)
user_dfs = {g: load_data(f"{USER_DIR}/{g}.csv") for g in GENRES}

# Professional reviews (RogerEbert)
pro_dfs = {g: load_data(f"{PRO_DIR}/{g}.csv") for g in GENRES}

for g in GENRES:
    print(f"{g:<10}  users: {len(user_dfs[g]):>4}   pros: {len(pro_dfs[g]):>3}")


## Preview

In [ ]:
user_dfs["adventure"].head()

In [ ]:
pro_dfs["adventure"].head()

## 1. Rating distributions across genres

In [ ]:
def plot_genre_distribution(pro_dfs, user_dfs, genres):
    plt.figure(figsize=(14, 7))
    for g in genres:
        sns.kdeplot(pro_dfs[g]["Pro Rating"], label=f"Pro — {g.title()}", fill=True, alpha=0.3)
    for g in genres:
        sns.kdeplot(user_dfs[g]["Rating"], label=f"User — {g.title()}", fill=True, alpha=0.3)
    plt.title("Rating Distributions Across Genres — Professionals vs. Users")
    plt.xlabel("Rating (0–10)")
    plt.ylabel("Density")
    plt.legend(loc="upper left", ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()

plot_genre_distribution(pro_dfs, user_dfs, GENRES)


## 2. Review volume comparison

Why this matters: RogerEbert has roughly one professional review per movie, whereas IMDb has hundreds of user reviews per movie. This asymmetry shapes how we interpret the statistics downstream.

In [ ]:
def compare_review_volume(pro_dfs, user_dfs, genres):
    pro_counts  = [len(pro_dfs[g])  for g in genres]
    user_counts = [len(user_dfs[g]) for g in genres]

    plt.figure(figsize=(10, 6))
    bar_width = 0.35
    index = range(len(genres))
    plt.bar(index, pro_counts, width=bar_width, label="Professional")
    plt.bar([i + bar_width for i in index], user_counts, width=bar_width, label="User")
    plt.xlabel("Genre")
    plt.ylabel("Number of reviews")
    plt.title("Review Volume: Professionals vs. Users")
    plt.xticks([i + bar_width / 2 for i in index], [g.title() for g in genres])
    plt.legend()
    plt.tight_layout()
    plt.show()

compare_review_volume(pro_dfs, user_dfs, GENRES)


## 3. Descriptive statistics

In [ ]:
def descriptive_statistics(pro_dfs, user_dfs, genres):
    all_stats = pd.DataFrame()
    for g in genres:
        pro_stats  = pro_dfs[g]["Pro Rating"].describe()
        user_stats = user_dfs[g]["Rating"].describe()
        stats_df = pd.DataFrame({"Professional": pro_stats, "User": user_stats})
        stats_df.columns = pd.MultiIndex.from_product([[g.title()], stats_df.columns])
        all_stats = stats_df if all_stats.empty else pd.concat([all_stats, stats_df], axis=1)
    return all_stats

descriptive_statistics(pro_dfs, user_dfs, GENRES)


## 4. Average ratings by genre

In [ ]:
def compare_average_ratings(pro_dfs, user_dfs, genres):
    averages_pro  = [pro_dfs[g]["Pro Rating"].astype(float).mean() for g in genres]
    averages_user = [user_dfs[g]["Rating"].astype(float).mean()    for g in genres]

    fig, ax = plt.subplots(figsize=(10, 6))
    bar_width = 0.35
    index = range(len(genres))
    ax.bar(index, averages_pro, width=bar_width, label="Professional", color="steelblue")
    ax.bar([i + bar_width for i in index], averages_user, width=bar_width, label="User", color="indianred")
    ax.set_xlabel("Genre")
    ax.set_ylabel("Average rating (0–10)")
    ax.set_title("Average Ratings by Genre — Professionals vs. Users")
    ax.set_xticks([i + bar_width / 2 for i in index])
    ax.set_xticklabels([g.title() for g in genres])
    ax.legend()
    plt.tight_layout()
    plt.show()

compare_average_ratings(pro_dfs, user_dfs, GENRES)


## 5. Sentiment distribution (TextBlob EDA baseline)

The charts below use TextBlob polarity as a fast sanity check. The formal sentiment classification (VADER + RoBERTa, with confusion matrices) lives in `02_sentiment_analysis.ipynb`.

In [ ]:
def sentiment_analysis(text):
    return TextBlob(str(text)).sentiment.polarity


def process_genre_data(pro_df, user_df, genre, ax1, ax2):
    pro_df  = pro_df.copy()
    user_df = user_df.copy()

    pro_df["Sentiment"]  = pro_df["Review"].apply(sentiment_analysis)
    user_df["Sentiment"] = user_df["Review Text"].apply(sentiment_analysis)

    merged = pro_df[["Title", "Pro Rating"]].merge(
        user_df[["Title", "Rating"]].drop_duplicates(subset=["Title"]),
        on="Title", suffixes=("_Pro", "_User"),
    )
    correlation = merged[["Pro Rating", "Rating"]].corr()

    sns.histplot(pro_df["Sentiment"],  color="blue", label="Professional", kde=True, ax=ax1, alpha=0.6)
    sns.histplot(user_df["Sentiment"], color="red",  label="User",         kde=True, ax=ax1, alpha=0.6)
    ax1.set_title(f"Sentiment Distribution — {genre.title()}")
    ax1.legend()

    sns.scatterplot(data=merged, x="Pro Rating", y="Rating", ax=ax2)
    ax2.set_title(f"Ratings Correlation — {genre.title()}")
    ax2.set_xlabel("Professional Ratings")
    ax2.set_ylabel("User Ratings")

    return correlation


n_genres = len(GENRES)
fig, axes = plt.subplots(n_genres, 2, figsize=(12, 3 * n_genres))

correlations = {}
for i, g in enumerate(GENRES):
    corr = process_genre_data(pro_dfs[g], user_dfs[g], g, axes[i][0], axes[i][1])
    correlations[g] = corr

plt.tight_layout()
plt.show()


## 6. Correlation summary

How strongly do average user ratings agree with professional ratings within each genre?

In [ ]:
summary = pd.DataFrame({
    "Genre": [g.title() for g in GENRES],
    "Pro / User correlation": [round(correlations[g].iloc[0, 1], 3) for g in GENRES],
})
summary


## Findings

- **Comedy has the strongest agreement** between professionals and audiences (r ≈ 0.63). Users and critics tend to converge on what's funny.
- **Superhero has the weakest agreement** (r ≈ 0.25). Critics evaluate craft and narrative; audiences evaluate entertainment and fandom service — these diverge.
- **User ratings cluster tighter** than professional ratings (smaller standard deviations across every genre). Critics are more willing to go to the extremes.
- **Users are more positive on average** for Adventure and Superhero; **professionals are more positive** for Comedy, Crime, and Horror.

Full discussion lives in the written report (`../report/final_report.pdf`).